In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# *Introduction to Hugging Face transformers and datasets*

*Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text.
What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51?
Note: We follow zero-indexing here.*

In [1]:
from datasets import load_dataset

# 1. Load the CSV file using Hugging Face datasets
# Replace the path if your train.csv is located in a different directory
file_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
dataset = load_dataset('csv', data_files=file_path, split='train')

# 2. Define the function to concatenate the columns
def add_combined_text(example):
    # Ensure both are treated as strings to avoid TypeErrors in case of missing data
    example['combined_text'] = str(example['prompt']) + " " + str(example['A'])
    return example

# 3. Use .map() to apply the function and create the new column
dataset = dataset.map(add_combined_text)

# 4. Access the row at index 51 (zero-indexed)
text_at_51 = dataset[51]['combined_text']

# 5. Calculate and print the exact character length
char_length = len(text_at_51)
print(f"The exact character length of combined_text at index 51 is: {char_length}")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

The exact character length of combined_text at index 51 is: 614


*Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?*

In [2]:
from transformers import AutoTokenizer

# 1. Initialize the bert-base-uncased tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Access the vocabulary size property
vocab_size = tokenizer.vocab_size

# 3. Print the result
print(f"The exact vocabulary size is: {vocab_size}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The exact vocabulary size is: 30522


*Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token*

In [3]:
from transformers import AutoTokenizer

# 1. Initialize the tokenizer (if not already done in your notebook)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Access the separator token ID property
sep_id = tokenizer.sep_token_id

# Alternatively, you can look it up by the string token itself:
# sep_id = tokenizer.convert_tokens_to_ids("[SEP]")

# 3. Print the result
print(f"The exact integer ID for the [SEP] token is: {sep_id}")

The exact integer ID for the [SEP] token is: 102


*Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 
What is the exact geometric shape (dimensions) of the resulting input_ids tensor?*

In [8]:
# Convert all prompts to strings to handle any missing (NaN) or non-string values
clean_prompts = [str(p) for p in dataset['prompt']]

# 1. Tokenize the cleaned list of prompts simultaneously
tokenized_outputs = tokenizer(
    clean_prompts, 
    padding='max_length', 
    truncation=True, 
    max_length=128, 
    return_tensors='pt'
)

# 2. Extract and print the geometric shape of the input_ids tensor
input_ids_shape = tokenized_outputs['input_ids'].shape
print(f"The exact geometric shape of the input_ids tensor is: {input_ids_shape}")

The exact geometric shape of the input_ids tensor is: torch.Size([2000, 128])


# *BERT/RoBERTa Architecture & Attention Mechanisms*

*A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 
In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?*

In [9]:
from transformers import AutoConfig

# 1. Load the configuration for bert-base-uncased
config = AutoConfig.from_pretrained("bert-base-uncased")

# 2. Calculate the head dimension using the config properties
head_dim = config.hidden_size // config.num_attention_heads

# 3. Print the result
print(f"Hidden Size: {config.hidden_size}")
print(f"Attention Heads: {config.num_attention_heads}")
print(f"Calculated Head Dimension: {head_dim}")

Hidden Size: 768
Attention Heads: 12
Calculated Head Dimension: 64


*Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 
What is the exact shape of the last_hidden_state tensor returned? 
Note: We follow zero-indexing here.*

In [10]:
from transformers import AutoModel, AutoTokenizer
import torch

# 1. Initialize both the tokenizer and the base BERT model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# 2. Extract the prompt at row index 0 (ensuring it's a string)
prompt_0 = str(dataset['prompt'][0])

# 3. Tokenize using default settings (no manual padding or truncation)
# We must use return_tensors='pt' to get PyTorch tensors for the model
inputs = tokenizer(prompt_0, return_tensors='pt')

# 4. Pass inputs through the model (wrapped in torch.no_grad() to save memory)
with torch.no_grad():
    outputs = model(**inputs)

# 5. Extract and print the shape of the last_hidden_state
last_hidden_state_shape = outputs.last_hidden_state.shape
print(f"The exact shape of the last_hidden_state tensor is: {last_hidden_state_shape}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The exact shape of the last_hidden_state tensor is: torch.Size([1, 31, 768])


*Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).*

In [11]:
from transformers import AutoModel, AutoTokenizer
import torch

# 1. Initialize tokenizer and model (if not already done)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# 2. Prepare the input from row ID 0
prompt_0 = str(dataset['prompt'][0])
inputs = tokenizer(prompt_0, return_tensors='pt')

# 3. Pass through the model to get last_hidden_state
with torch.no_grad():
    outputs = model(**inputs)
    last_hidden_state = outputs.last_hidden_state

# 4. Extract the [CLS] embedding vector (Batch 0, Token Index 0, All Hidden Dimensions)
cls_vector = last_hidden_state[0, 0, :]

# 5. Extract the first 5 float values from the [CLS] vector
first_5_values = cls_vector[:5]

# 6. Compute the sum and round to 4 decimal places
sum_first_5 = torch.sum(first_5_values).item()
rounded_sum = round(sum_first_5, 4)

print(f"The first 5 values are: {first_5_values.tolist()}")
print(f"The exact sum rounded to 4 decimal places is: {rounded_sum}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The first 5 values are: [-0.4676641821861267, -0.07544498145580292, -0.20190085470676422, -0.007064265664666891, -0.4480222165584564]
The exact sum rounded to 4 decimal places is: -1.2001


*Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 
What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places)*

In [12]:
from transformers import AutoModel, AutoTokenizer
import torch

# 1. Initialize tokenizer and model with output_attentions=True
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

# 2. Tokenize the exact string
text = "Light-ion fusion is a technique."
inputs = tokenizer(text, return_tensors='pt')

# 3. Programmatically find the specific token index for the word "fusion"
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
fusion_index = tokens.index("fusion")

# 4. Pass inputs through the model
with torch.no_grad():
    outputs = model(**inputs)

# 5. Extract attention from the last layer (index -1) and first head (index 0)
# outputs.attentions shape: (batch_size, num_heads, sequence_length, sequence_length)
last_layer_attention = outputs.attentions[-1] 
first_head_attention = last_layer_attention[0, 0, :, :] 

# 6. Get the weight that [CLS] (index 0) pays to "fusion" (fusion_index)
attention_weight = first_head_attention[0, fusion_index].item()
rounded_weight = round(attention_weight, 4)

print(f"Tokenized Sequence: {tokens}")
print(f"Token Index for 'fusion': {fusion_index}")
print(f"The exact attention weight rounded to 4 decimal places is: {rounded_weight}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenized Sequence: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Token Index for 'fusion': 4
The exact attention weight rounded to 4 decimal places is: 0.1025


# *Context-Aware Embeddings*

*Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places?
Note: We follow zero-indexing here.*

In [13]:
# Install the library if it isn't already installed in your environment
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer, util

# 1. Initialize the sentence-transformers model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# 2. Extract the prompt and Option B from row ID 0 (ensuring they are strings)
prompt_0 = str(dataset['prompt'][0])
option_b_0 = str(dataset['B'][0])  # Adjust to 'Option B' if your column name is spelled out

# 3. Generate embeddings using the .encode() method
prompt_embedding = model.encode(prompt_0, convert_to_tensor=True)
option_b_embedding = model.encode(option_b_0, convert_to_tensor=True)

# 4. Calculate cosine similarity using util.cos_sim()
similarity_tensor = util.cos_sim(prompt_embedding, option_b_embedding)

# 5. Extract the float value and round to 4 decimal places
similarity_score = similarity_tensor.item()
rounded_score = round(similarity_score, 4)

print(f"Prompt: {prompt_0}")
print(f"Option B: {option_b_0}")
print(f"The resulting similarity score rounded to 4 decimal places is: {rounded_score}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Option B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
The resulting similarity score rounded to 4 decimal places is: 0.7658


*Build two complete ranking pipelines evaluating every row in train.csv.
Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.
Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.
First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 
Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?*

In [14]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import torch

# 1. Prepare data arrays (ensuring all entries are treated as strings)
options_labels = ['A', 'B', 'C', 'D', 'E']
prompts = [str(row['prompt']) for row in dataset]
answers = [str(row['answer']) for row in dataset]  # Adjust to 'label' if your column is named differently

options_data = {label: [str(row[label]) for row in dataset] for label in options_labels}

# 2. Pipeline 1: Global TF-IDF Setup
print("Fitting TF-IDF Vectorizer...")
all_text = prompts.copy()
for label in options_labels:
    all_text.extend(options_data[label])

tfidf = TfidfVectorizer()
tfidf.fit(all_text)

# 3. Pipeline 2: MiniLM Embedding Generation
print("Generating MiniLM Embeddings...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

prompt_embeds = model.encode(prompts, convert_to_tensor=True, show_progress_bar=True)
option_embeds = {}
for label in options_labels:
    option_embeds[label] = model.encode(options_data[label], convert_to_tensor=True, show_progress_bar=True)

# 4. Evaluation Loop
minilm_ap_scores = []
conditional_question_count = 0

print("Evaluating rows...")
for i in range(len(dataset)):
    ans = answers[i]
    p_emb = prompt_embeds[i]
    
    # --- TF-IDF Top 3 Calculation ---
    p_tfidf = tfidf.transform([prompts[i]])
    tfidf_sims = {}
    for label in options_labels:
        o_tfidf = tfidf.transform([options_data[label][i]])
        tfidf_sims[label] = cosine_similarity(p_tfidf, o_tfidf)[0][0]
    tfidf_ranked = sorted(tfidf_sims, key=tfidf_sims.get, reverse=True)[:3]
    
    # --- MiniLM Top 3 Calculation ---
    minilm_sims = {}
    for label in options_labels:
        o_emb = option_embeds[label][i]
        minilm_sims[label] = util.cos_sim(p_emb, o_emb).item()
    minilm_ranked = sorted(minilm_sims, key=minilm_sims.get, reverse=True)[:3]
    
    # --- Calculate MAP@3 for MiniLM ---
    if ans in minilm_ranked:
        rank = minilm_ranked.index(ans) + 1
        minilm_ap_scores.append(1.0 / rank)
    else:
        minilm_ap_scores.append(0.0)
        
    # --- Comparative Counter ---
    # Correct answer NOT in TF-IDF Top-3 BUT IS in MiniLM Top-3
    if (ans not in tfidf_ranked) and (ans in minilm_ranked):
        conditional_question_count += 1

# 5. Output Results
final_map3 = np.mean(minilm_ap_scores)

print("\n" + "="*40)
print(f"1. Final MiniLM MAP@3 Score: {round(final_map3, 4)}")
print(f"2. Exact Count of Mismatched Right Answers: {conditional_question_count}")
print("="*40)

Fitting TF-IDF Vectorizer...
Generating MiniLM Embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Evaluating rows...

1. Final MiniLM MAP@3 Score: 0.4231
2. Exact Count of Mismatched Right Answers: 531


# *Zero-shot classification concepts*

*Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places)*

In [15]:
from transformers import pipeline
import torch

# 1. Initialize the zero-shot classification pipeline
# It automatically defaults to 'facebook/bart-large-mnli'
# Using device=0 runs it on GPU if available in your Kaggle session
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

# 2. Extract the prompt and options A, B, and C from the 2nd row (index 1)
row_1 = dataset[1]
prompt_1 = str(row_1['prompt'])
candidate_options = [str(row_1['A']), str(row_1['B']), str(row_1['C'])]

# 3. Perform zero-shot classification
results = classifier(prompt_1, candidate_labels=candidate_options)

# 4. The pipeline returns 'labels' and 'scores' pre-sorted from highest to lowest probability
top_option = results['labels'][0]
top_score = results['scores'][0]
rounded_score = round(top_score, 4)

# 5. Print results
print(f"Prompt: {prompt_1}\n")
print(f"Top-Ranked Option: {top_option}")
print(f"The probability score given to the top-ranked option is: {rounded_score}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Prompt: What is accelerator-based light-ion fusion?

Top-Ranked Option: Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.
The probability score given to the top-ranked option is: 0.4575


*Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 
What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?*

In [20]:
from transformers import pipeline
import torch

# 1. Initialize the zero-shot classification pipeline
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

# 2. Extract the prompt and options A, B, and C from the 2nd row (index 1)
row_1 = dataset[1]
prompt_1 = str(row_1['prompt'])
candidate_options = [str(row_1['A']), str(row_1['B']), str(row_1['C'])]

# 3. Run classification with multi_label=False (uses Softmax)
results_softmax = classifier(prompt_1, candidate_labels=candidate_options, multi_label=False)
sum_softmax = sum(results_softmax['scores'])

# 4. Run classification with multi_label=True (uses independent Sigmoids)
results_sigmoid = classifier(prompt_1, candidate_labels=candidate_options, multi_label=True)
sum_sigmoid = sum(results_sigmoid['scores'])

# 5. Calculate the absolute difference
abs_diff = abs(sum_softmax - sum_sigmoid)
rounded_diff = round(abs_diff, 4)

# 6. Print the breakdown and final answer
print(f"Sum of Softmax probabilities (multi_label=False): {sum_softmax:.4f}")
print(f"Sum of Sigmoid probabilities (multi_label=True): {sum_sigmoid:.4f}")
print(f"The absolute difference rounded to 4 decimal places is: {rounded_diff}")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Sum of Softmax probabilities (multi_label=False): 1.0000
Sum of Sigmoid probabilities (multi_label=True): 0.0005
The absolute difference rounded to 4 decimal places is: 0.9995


*Let's try Generative AI instead of Classification. 
Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model?*

In [22]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# 1. Load tokenizer and model explicitly for sequence-to-sequence generation
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Automatically utilize GPU if available in your Kaggle environment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 2. Extract prompt, Option A, and Option B from row index 0
row_0 = dataset[0]
prompt_0 = str(row_0['prompt'])
option_a = str(row_0['A'])
option_b = str(row_0['B'])

# 3. Construct the exact prompt string specified
input_text = f"Question: {prompt_0}. Is the correct answer A: {option_a} or B: {option_b}? Answer with just the letter A or B."

# 4. Tokenize the input text and move the tensors to the correct device
inputs = tokenizer(input_text, return_tensors="pt").to(device)

# 5. Generate output text tokens setting max_new_tokens=5
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=5)

# 6. Decode the tokens back to a string, skipping special tokens (like <pad> or </s>)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"The exact string output returned by the model is: '{generated_text}'")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

The exact string output returned by the model is: 'B'
